In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
torch.manual_seed(1337)
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [3]:
with open("../data/dataset.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [4]:
chars = list(sorted(set(text)))
vocab_size = len(chars)

In [5]:
stoi, itos = {}, {}
for i, ch in enumerate(chars):
    stoi[ch] = i
    itos[i] = ch

In [6]:
def get_batch(data, batch_size, seq_len):
    high = len(data) - seq_len
    ix = torch.randint(0, high, (batch_size, ), device=device)

    idx = ix[:, None] + torch.arange(seq_len, device=device)
    x = data[idx]
    y = data[idx+1]

    return x, y

In [7]:
class LSTMCell(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.gates = nn.Linear(vocab_size + hidden_size, 4 * hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.gates(combined)

        # Split into the four gate pre-activations (f, i, c, o)
        f, i, c, o = gates.chunk(4, dim=1)
        
        forget_gate = torch.sigmoid(f)
        input_gate = torch.sigmoid(i)
        candidate_memory = torch.tanh(c)
        output_gate = torch.sigmoid(o)

        cell_state = forget_gate * c_prev + input_gate * candidate_memory
        hidden = output_gate * torch.tanh(cell_state)

        return hidden, cell_state

In [8]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.lstm = LSTMCell(vocab_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None, c=None):
        batch_size, seq_len = x.shape

        if h is None:
            h = torch.zeros((batch_size, self.hidden_size), device=device)

        if c is None:
            c = torch.zeros((batch_size, self.hidden_size), device=device)

        outputs = []
        for t in range(seq_len):
            x_t = x[:, t]
            # One-hot encoding
            x_t = F.one_hot(x_t, num_classes=self.vocab_size).float()

            h, c = self.lstm(x_t, h, c)
            logits = self.fc(h)

            outputs.append(logits)

        outputs = torch.stack(outputs, dim=1)
        return outputs, h, c

In [9]:
hidden_size = 256
model = CharLSTM(vocab_size, hidden_size).to(device)

In [10]:
def loss_fn(logits, targets):
    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    targets = targets.view(B*T)

    # Flatten Batch and Time
    loss = F.cross_entropy(logits, targets)
    return loss

epochs = 20000
batch_size = 128
seq_len = 64
lr = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long, device=device)

model.train()
for epoch in range(epochs):
    x, y = get_batch(data, batch_size, seq_len)
    optimizer.zero_grad()
    
    logits, _, _ = model(x)
    loss = loss_fn(logits, y)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    optimizer.step()

    if epoch % 2000 == 0:
        print(f"Epoch: {epoch}, Loss: {loss.item()}")

print(f"Epoch: {epochs}, Loss: {loss.item()}")

Epoch: 0, Loss: 4.168654918670654
Epoch: 2000, Loss: 1.7936228513717651
Epoch: 4000, Loss: 1.56986403465271
Epoch: 6000, Loss: 1.4639151096343994
Epoch: 8000, Loss: 1.3906469345092773
Epoch: 10000, Loss: 1.3206112384796143
Epoch: 12000, Loss: 1.2438298463821411
Epoch: 14000, Loss: 1.2742798328399658
Epoch: 16000, Loss: 1.2565809488296509
Epoch: 18000, Loss: 1.2407522201538086
Epoch: 20000, Loss: 1.2392704486846924


In [11]:
@torch.no_grad()
def generate(start_text, max_tokens, temperature):
    model.eval()

    output = []
    output.append(start_text)
    x = torch.tensor([[stoi[ch] for ch in start_text]], dtype=torch.long, device=device)

    logits, h, c = model(x)
    
    for _ in range(max_tokens):
        # Use the last prediction
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=1)
        
        # Sample using multinomial distribution
        next_char_id = torch.multinomial(probs, num_samples=1)
        
        logits, h, c = model(next_char_id, h, c)
        output.append(itos[next_char_id.item()])

    return "".join(output)

In [12]:
result = generate("the", 100, 0.8)
print(result)

the way
To out our knees.

FRIAR JOHN:
I pray you in the town and true tribunes
Being done a made the t
